# S3 Winner Subset Characterization — SPARC Rotation Curves

**Symonic LLC / MCORE-1 research stack (exploratory)**

This notebook extends the binary (baryons + NFW) vs ternary (baryons + NFW + disk-coupled S₃ Gaussian in V²) experiment.

## Scope and ethics

- **Do not** claim new dark matter physics. S₃ is treated as a **candidate structured residual** in velocity space.
- Adversarial controls and alternative halos test whether ternary wins **survive** beyond disk anchoring and NFW misfit.
- Outputs are **diagnostic**: use for internal science and grant supplements, not as final cosmological conclusions without peer review.

## Questions

1. Do ternary-winning galaxies form a coherent physical/statistical class?
2. Does S₃ capture disk-coupled structure, or mainly absorb poor NFW fits?
3. Verdict codes **A–D** (see final section).

In [ ]:
# Configuration
import os
import io
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy import stats

warnings.filterwarnings("ignore", category=RuntimeWarning)

SPARC_URL = "https://astroweb.case.edu/SPARC/Rotmod_LTG.zip"
DATA_DIR = os.environ.get("SPARC_DATA_DIR", "sparc_data")
OUT_DIR = Path("outputs_s3_characterization")
OUT_DIR.mkdir(exist_ok=True)

RNG = np.random.default_rng(42)
WINNER_DBIC = -2.0
LOSER_DBIC = 2.0

print("OUT_DIR =", OUT_DIR.resolve())

In [ ]:
def download_sparc():
    if os.path.exists(DATA_DIR):
        n = sum(1 for _root, _d, fs in os.walk(DATA_DIR) for _f in fs)
        if n > 100:
            print(f"Using existing SPARC tree ({n} files under {DATA_DIR})")
            return
    print("Downloading SPARC Rotmod_LTG...")
    import urllib.request

    os.makedirs(DATA_DIR, exist_ok=True)
    with urllib.request.urlopen(SPARC_URL, timeout=120) as resp:
        z = io.BytesIO(resp.read())
    with zipfile.ZipFile(z) as zf:
        zf.extractall(DATA_DIR)
    print("Extracted to", DATA_DIR)


def parse_galaxy(filepath):
    data = {"Rad": [], "Vobs": [], "errV": [], "Vgas": [], "Vdisk": [], "Vbul": []}
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or line.startswith("!"):
                continue
            parts = line.split()
            if len(parts) < 6:
                continue
            try:
                vals = [float(p) for p in parts[:6]]
                data["Rad"].append(vals[0])
                data["Vobs"].append(vals[1])
                data["errV"].append(max(vals[2], 1.0))
                data["Vgas"].append(vals[3])
                data["Vdisk"].append(vals[4])
                data["Vbul"].append(vals[5])
            except ValueError:
                continue
    return {k: np.array(v, dtype=float) for k, v in data.items()}


def load_all_galaxies(data_dir=DATA_DIR, min_points=8):
    galaxies = {}
    roots = {data_dir}
    for root, _dirs, files in os.walk(data_dir):
        roots.add(root)
    for root in sorted(roots):
        for fname in sorted(os.listdir(root)):
            fpath = os.path.join(root, fname)
            if not os.path.isfile(fpath):
                continue
            try:
                gal = parse_galaxy(fpath)
                if len(gal["Rad"]) >= min_points:
                    name = fname.replace("_rotmod.dat", "").replace(".dat", "")
                    galaxies[name] = gal
            except Exception:
                continue
    return galaxies


download_sparc()
galaxies = load_all_galaxies()
print(f"Loaded {len(galaxies)} galaxies with >= 8 points")

In [ ]:
# --- Same physics as S3_Cosmological_Crystallization_Analysis.ipynb ---

def V_baryonic(gal, Ydisk=0.5, Ybul=0.7):
    Vd2 = Ydisk * np.sign(gal["Vdisk"]) * gal["Vdisk"] ** 2
    Vg2 = np.sign(gal["Vgas"]) * gal["Vgas"] ** 2
    Vb2 = Ybul * np.sign(gal["Vbul"]) * gal["Vbul"] ** 2
    return Vd2 + Vg2 + Vb2


def V_NFW_squared(r, Vscale2, rs):
    x = np.maximum(r / rs, 1e-10)
    fx = np.log(1 + x) - x / (1 + x)
    return Vscale2 * fx / x


def V_S3_squared(r, amp, r_peak, sigma_g):
    """Additive V^2 bump (Gaussian in radius). amp is linear scale (as in original notebook)."""
    return amp * np.exp(-0.5 * ((r - r_peak) / np.maximum(sigma_g, 1e-6)) ** 2)


def burkert_f(x):
    return 0.5 * np.log(1 + x * x) + np.arctan(x) - np.log(np.maximum(1 + x, 1e-30))


def V2_burkert(r, A, r0):
    """Two-parameter Burkert-style halo contribution to V^2 (A absorbs 4πGρ₀r₀³ scaling)."""
    x = r / np.maximum(r0, 1e-6)
    return A * burkert_f(x) / np.maximum(r, 1e-6)


def Vtot_from_V2(Vbar2, Vhalo2, Vs32=None):
    Vtot2 = Vbar2 + Vhalo2
    if Vs32 is not None:
        Vtot2 = Vtot2 + Vs32
    return np.sign(Vtot2) * np.sqrt(np.abs(Vtot2))


def Vbar_signed(Vbar2):
    return np.sign(Vbar2) * np.sqrt(np.abs(Vbar2))


def disk_scale_length(gal):
    r = gal["Rad"]
    idx = int(np.argmax(np.abs(gal["Vdisk"])))
    r_peak_disk = r[idx]
    return max(r_peak_disk / 2.15, 0.5), r_peak_disk


def fit_binary(gal, Ydisk=0.5, Ybul=0.7):
    r, Vobs, errV = gal["Rad"], gal["Vobs"], gal["errV"]
    Vbar2 = V_baryonic(gal, Ydisk, Ybul)

    def residuals(p):
        log_Vscale2, log_rs = p
        Vscale2, rs = 10 ** log_Vscale2, 10 ** log_rs
        Vnfw2 = V_NFW_squared(r, Vscale2, rs)
        Vtot = Vtot_from_V2(Vbar2, Vnfw2)
        return (Vtot - Vobs) / errV

    x0 = [4.0, 0.5]
    bounds = ([1, -1], [7, 2.5])
    try:
        res = least_squares(residuals, x0, bounds=bounds, method="trf", max_nfev=5000)
        if not res.success:
            return None
        Vscale2, rs = 10 ** res.x[0], 10 ** res.x[1]
        Vnfw2 = V_NFW_squared(r, Vscale2, rs)
        Vbin = Vtot_from_V2(Vbar2, Vnfw2)
        chi2 = float(np.sum(res.fun**2))
        n = len(r)
        k = 2
        bic = chi2 + k * np.log(n)
        aic = chi2 + 2 * k
        return {
            "success": True,
            "chi2": chi2,
            "bic": bic,
            "aic": aic,
            "n": n,
            "k": k,
            "Vscale2": Vscale2,
            "rs": rs,
            "Vbar2": Vbar2,
            "Vnfw2": Vnfw2,
            "Vbin": Vbin,
            "residual_bin": Vbin - Vobs,
        }
    except Exception:
        return None


def fit_ternary_disk_coupled(gal, Ydisk=0.5, Ybul=0.7, r_s3=None, sigma_s3=None):
    """Ternary with fixed (r_s3, sigma_s3) from disk; default = original coupling."""
    r, Vobs, errV = gal["Rad"], gal["Vobs"], gal["errV"]
    Vbar2 = V_baryonic(gal, Ydisk, Ybul)
    h_R, _ = disk_scale_length(gal)
    if r_s3 is None:
        r_s3 = 2.15 * h_R
    if sigma_s3 is None:
        sigma_s3 = h_R

    def residuals(p):
        log_Vscale2, log_rs, log_Vs3 = p
        Vscale2, rs, Vs3 = 10 ** log_Vscale2, 10 ** log_rs, 10 ** log_Vs3
        Vnfw2 = V_NFW_squared(r, Vscale2, rs)
        Vs32 = V_S3_squared(r, Vs3, r_s3, sigma_s3)
        Vtot = Vtot_from_V2(Vbar2, Vnfw2, Vs32)
        return (Vtot - Vobs) / errV

    x0 = [4.0, 0.5, 3.0]
    bounds = ([1, -1, 0], [7, 2.5, 7])
    try:
        res = least_squares(residuals, x0, bounds=bounds, method="trf", max_nfev=5000)
        if not res.success:
            return None
        Vscale2, rs, Vs3 = 10 ** res.x[0], 10 ** res.x[1], 10 ** res.x[2]
        Vnfw2 = V_NFW_squared(r, Vscale2, rs)
        Vs32 = V_S3_squared(r, Vs3, r_s3, sigma_s3)
        Vter = Vtot_from_V2(Vbar2, Vnfw2, Vs32)
        chi2 = float(np.sum(res.fun**2))
        n = len(r)
        k = 3
        bic = chi2 + k * np.log(n)
        aic = chi2 + 2 * k
        return {
            "success": True,
            "chi2": chi2,
            "bic": bic,
            "aic": aic,
            "n": n,
            "k": k,
            "Vscale2": Vscale2,
            "rs": rs,
            "Vs3": Vs3,
            "h_R": h_R,
            "r_s3": r_s3,
            "sigma_s3": sigma_s3,
            "Vbar2": Vbar2,
            "Vnfw2": Vnfw2,
            "Vs32": Vs32,
            "Vter": Vter,
            "residual_ter": Vter - Vobs,
        }
    except Exception:
        return None


def fit_ternary_free_gaussian(gal, Ydisk=0.5, Ybul=0.7):
    """NFW + baryons + Gaussian with FREE log peak and log width (5 free params)."""
    r, Vobs, errV = gal["Rad"], gal["Vobs"], gal["errV"]
    Vbar2 = V_baryonic(gal, Ydisk, Ybul)
    Rmax = float(np.max(r))

    def residuals(p):
        log_Vscale2, log_rs, log_Vs3, log_rpk, log_sig = p
        Vscale2, rs = 10 ** log_Vscale2, 10 ** log_rs
        Vs3 = 10 ** log_Vs3
        rpk = 10 ** log_rpk
        sig = 10 ** log_sig
        Vnfw2 = V_NFW_squared(r, Vscale2, rs)
        Vs32 = V_S3_squared(r, Vs3, rpk, sig)
        Vtot = Vtot_from_V2(Vbar2, Vnfw2, Vs32)
        return (Vtot - Vobs) / errV

    h_R, _ = disk_scale_length(gal)
    x0 = [4.0, 0.5, 3.0, np.log10(max(0.5 * Rmax, 0.5)), np.log10(max(h_R, 0.2))]
    bounds = ([1, -1, 0, -1, -1], [7, 2.5, 7, np.log10(Rmax * 2), np.log10(Rmax)])
    try:
        res = least_squares(residuals, x0, bounds=bounds, method="trf", max_nfev=8000)
        if not res.success:
            return None
        chi2 = float(np.sum(res.fun**2))
        n, k = len(r), 5
        return {"success": True, "chi2": chi2, "bic": chi2 + k * np.log(n), "aic": chi2 + 2 * k, "k": k, "n": n}
    except Exception:
        return None


def fit_ternary_null_s3(gal, Ydisk=0.5, Ybul=0.7):
    """Same as ternary but Vs3 fixed to 0 (3 params with one degenerate — should match binary quality)."""
    r, Vobs, errV = gal["Rad"], gal["Vobs"], gal["errV"]
    Vbar2 = V_baryonic(gal, Ydisk, Ybul)
    h_R, _ = disk_scale_length(gal)
    r_s3, sigma_s3 = 2.15 * h_R, h_R

    def residuals(p):
        log_Vscale2, log_rs = p
        Vscale2, rs = 10 ** log_Vscale2, 10 ** log_rs
        Vnfw2 = V_NFW_squared(r, Vscale2, rs)
        Vs32 = np.zeros_like(r)
        Vtot = Vtot_from_V2(Vbar2, Vnfw2, Vs32)
        return (Vtot - Vobs) / errV

    x0 = [4.0, 0.5]
    bounds = ([1, -1], [7, 2.5])
    try:
        res = least_squares(residuals, x0, bounds=bounds, method="trf", max_nfev=5000)
        if not res.success:
            return None
        chi2 = float(np.sum(res.fun**2))
        n, k = len(r), 2  # effective DOF matches binary (Vs3 fixed at 0)
        return {"success": True, "chi2": chi2, "bic": chi2 + k * np.log(n), "aic": chi2 + 2 * k, "k": k, "n": n}
    except Exception:
        return None


def fit_burkert(gal, Ydisk=0.5, Ybul=0.7):
    """Baryons + Burkert halo (2 free params)."""
    r, Vobs, errV = gal["Rad"], gal["Vobs"], gal["errV"]
    Vbar2 = V_baryonic(gal, Ydisk, Ybul)

    def residuals(p):
        log_A, log_r0 = p
        A, r0 = 10 ** log_A, 10 ** log_r0
        Vbur2 = V2_burkert(r, A, r0)
        Vtot = Vtot_from_V2(Vbar2, Vbur2)
        return (Vtot - Vobs) / errV

    x0 = [4.0, 0.5]
    bounds = ([1, -1], [8, 2.5])
    try:
        res = least_squares(residuals, x0, bounds=bounds, method="trf", max_nfev=5000)
        if not res.success:
            return None
        chi2 = float(np.sum(res.fun**2))
        n, k = len(r), 2
        return {"success": True, "chi2": chi2, "bic": chi2 + k * np.log(n), "aic": chi2 + 2 * k, "k": k, "n": n}
    except Exception:
        return None


def component_Vsquared(V2):
    """Display convention: signed sqrt for additive V^2 components (can be negative for bar)."""
    return np.sign(V2) * np.sqrt(np.abs(V2))

In [ ]:
# --- Run fits for all galaxies; store rows + full binary/ternary dicts for plotting ---

rows = []
fit_store = {}

for name in sorted(galaxies.keys()):
    gal = galaxies[name]
    b = fit_binary(gal)
    t = fit_ternary_disk_coupled(gal)
    if b is None or t is None:
        continue
    delta_bic = t["bic"] - b["bic"]
    if delta_bic < WINNER_DBIC:
        winner = "TERNARY"
    elif delta_bic > LOSER_DBIC:
        winner = "BINARY"
    else:
        winner = "TIE"

    Rmax = float(np.max(gal["Rad"]))
    Vmax = float(np.max(np.abs(gal["Vobs"])))
    Vg2 = np.sign(gal["Vgas"]) * gal["Vgas"] ** 2
    Vd2 = np.sign(gal["Vdisk"]) * gal["Vdisk"] ** 2
    gas_proxy = float(np.nanmean(np.abs(Vg2) / (np.abs(Vd2) + np.abs(Vg2) + 1e-6)))
    disk_dom = float(np.nanmean(np.abs(Vd2) / (np.abs(Vd2) + np.abs(Vg2) + 1e-6)))

    row = {
        "name": name,
        "n_points": b["n"],
        "binary_chi2": b["chi2"],
        "ternary_chi2": t["chi2"],
        "binary_bic": b["bic"],
        "ternary_bic": t["bic"],
        "binary_aic": b["aic"],
        "ternary_aic": t["aic"],
        "delta_bic": delta_bic,
        "winner": winner,
        "Rmax": Rmax,
        "Vmax": Vmax,
        "gas_frac_proxy": gas_proxy,
        "disk_dom_proxy": disk_dom,
        "r_s3": t["r_s3"],
        "sigma_s3": t["sigma_s3"],
        "r_s3_over_Rmax": t["r_s3"] / Rmax,
        "sigma_s3_over_Rmax": t["sigma_s3"] / Rmax,
        "Vs3": t["Vs3"],
        "Vs3_over_Vmaxsq": t["Vs3"] / (Vmax**2 + 1e-6),
        "chi2_bin_per_n": b["chi2"] / b["n"],
        "chi2_ter_per_n": t["chi2"] / t["n"],
        "delta_chi2": t["chi2"] - b["chi2"],
        "rs_bin": b["rs"],
        "rs_ter": t["rs"],
    }
    rows.append(row)
    fit_store[name] = {"gal": gal, "binary": b, "ternary": t}

df = pd.DataFrame(rows)
df.to_csv(OUT_DIR / "per_galaxy_fits.csv", index=False)
print("Saved", OUT_DIR / "per_galaxy_fits.csv", "rows:", len(df))
print(df["winner"].value_counts())
print("Median ΔBIC:", df["delta_bic"].median(), "Mean:", df["delta_bic"].mean())

In [ ]:
# --- Adversarial controls + extra baselines (per galaxy) ---

adv_rows = []
base_rows = []

for name in sorted(fit_store.keys()):
    gal = galaxies[name]
    b = fit_store[name]["binary"]
    t = fit_store[name]["ternary"]
    Rmax = float(np.max(gal["Rad"]))
    h_R, _ = disk_scale_length(gal)

    # 1) Random S3 anchor (same param count as ternary)
    r_rand = float(RNG.uniform(0.15 * Rmax, 0.95 * Rmax))
    sig_rand = float(RNG.uniform(max(0.2 * h_R, 0.2), min(2.0 * h_R, Rmax)))
    tr = fit_ternary_disk_coupled(gal, r_s3=r_rand, sigma_s3=sig_rand)

    # 2) Mirrored / wrong anchor: reflect disk peak across outer radius
    r_bad = float(max(0.5 * Rmax, 2 * Rmax - t["r_s3"]))
    tm = fit_ternary_disk_coupled(gal, r_s3=r_bad, sigma_s3=t["sigma_s3"])

    tf = fit_ternary_free_gaussian(gal)
    tn = fit_ternary_null_s3(gal)
    bur = fit_burkert(gal)

    adv_rows.append(
        {
            "name": name,
            "delta_bic_true": t["bic"] - b["bic"],
            "delta_bic_random_anchor": (tr["bic"] - b["bic"]) if tr else np.nan,
            "delta_bic_mirrored_anchor": (tm["bic"] - b["bic"]) if tm else np.nan,
            "delta_bic_free_gauss": (tf["bic"] - b["bic"]) if tf else np.nan,
            "delta_bic_null_s3_penalized": (tn["bic"] - b["bic"]) if tn else np.nan,
        }
    )
    base_rows.append(
        {
            "name": name,
            "bic_NFW_binary": b["bic"],
            "bic_NFW_ternary_disk": t["bic"],
            "bic_Burkert": bur["bic"] if bur else np.nan,
            "chi2_Burkert": bur["chi2"] if bur else np.nan,
        }
    )

adv_df = pd.DataFrame(adv_rows)
adv_df.to_csv(OUT_DIR / "adversarial_controls.csv", index=False)
base_df = pd.DataFrame(base_rows)
base_df.to_csv(OUT_DIR / "baseline_halos.csv", index=False)

print("Adversarial ΔBIC vs binary (median):")
for c in adv_df.columns:
    if c.startswith("delta_bic") and c != "delta_bic_true":
        print(f"  {c:32s} {adv_df[c].median():+.2f}")

print("\nInterpretation sketch:")
print("- If random/mirrored anchors often beat disk coupling → coupling not special.")
print("- If free Gaussian beats disk-ternary with lower BIC → shape not unique to disk anchor.")
print("- If Burkert beats both NFW models often → wins may be NFW misfit not S3 structure.")

In [ ]:
# --- Permutation: shuffle disk anchors among galaxies, refit ternary ---

names = list(fit_store.keys())
perm_deltas = []
n_perm = min(199, max(49, len(names) * 2))  # cap runtime

true_deltas = np.array([fit_store[n]["ternary"]["bic"] - fit_store[n]["binary"]["bic"] for n in names])

for _ in range(n_perm):
    perm = RNG.permutation(len(names))
    shuf = []
    for i, n in enumerate(names):
        donor = names[perm[i]]
        r_s3_d = fit_store[donor]["ternary"]["r_s3"]
        sig_d = fit_store[donor]["ternary"]["sigma_s3"]
        gal = galaxies[n]
        tt = fit_ternary_disk_coupled(gal, r_s3=r_s3_d, sigma_s3=sig_d)
        bb = fit_store[n]["binary"]
        if tt:
            shuf.append(tt["bic"] - bb["bic"])
    if shuf:
        perm_deltas.append(np.median(shuf))

perm_deltas = np.array(perm_deltas)
obs_median = np.median(true_deltas)
p_perm = float(np.mean(perm_deltas <= obs_median))

plt.figure(figsize=(8, 4))
plt.hist(perm_deltas, bins=30, alpha=0.75, label=f"Permuted anchors (n={n_perm})")
plt.axvline(obs_median, color="crimson", lw=2, label=f"Observed median ΔBIC={obs_median:.2f}")
plt.xlabel("Median ΔBIC (ternary_shuffled − binary) across sample")
plt.ylabel("Count")
plt.legend()
plt.title("Permutation null: S3 anchors shuffled across galaxies")
plt.tight_layout()
plt.savefig(OUT_DIR / "permutation_median_delta_bic.png", dpi=150)
plt.show()

print(f"Observed median ΔBIC: {obs_median:.3f}")
print(f"Permutation p-value (one-sided, lower is more ternary-favorable): {p_perm:.4f}")

In [ ]:
def plot_galaxy_decomposition(name, out_path, title_suffix=""):
    pack = fit_store[name]
    gal, b, t = pack["gal"], pack["binary"], pack["ternary"]
    r = gal["Rad"]
    Vobs, errV = gal["Vobs"], gal["errV"]

    Vbar = Vbar_signed(b["Vbar2"])
    Vnfw_b = component_Vsquared(b["Vnfw2"])
    Vbin = b["Vbin"]

    Vnfw_t = component_Vsquared(t["Vnfw2"])
    Vs3 = component_Vsquared(t["Vs32"])
    Vter = t["Vter"]

    fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
    ax0, ax1 = axes

    ax0.errorbar(r, Vobs, yerr=errV, fmt="ko", ms=3, capsize=2, label="V_obs")
    ax0.plot(r, Vbar, "g--", lw=1.5, label="V_bar (signed sqrt)")
    ax0.plot(r, Vnfw_b, "b:", lw=1.2, label="NFW (binary fit)")
    ax0.plot(r, Vnfw_t, "b-", lw=1.2, label="NFW (ternary fit)")
    ax0.plot(r, Vs3, "m-", lw=1.5, label="S₃ bump (V)")
    ax0.plot(r, Vbin, "c--", lw=1.2, label="Binary total")
    ax0.plot(r, Vter, "r-", lw=1.5, label="Ternary total")
    ax0.set_ylabel("km/s")
    ax0.set_title(f"{name} {title_suffix}")
    ax0.legend(loc="best", fontsize=8)

    ax1.axhline(0, color="k", lw=0.5)
    ax1.plot(r, b["residual_bin"], "c--", label="Residual binary")
    ax1.plot(r, t["residual_ter"], "r-", label="Residual ternary")
    ax1.set_xlabel("R [kpc]")
    ax1.set_ylabel("V - V_obs")
    ax1.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


top_ter = df.sort_values("delta_bic").head(10)["name"].tolist()
top_bin = df.sort_values("delta_bic", ascending=False).head(10)["name"].tolist()

for n in top_ter:
    plot_galaxy_decomposition(n, OUT_DIR / f"decomp_TERNARY_{n}.png", "(top ternary ΔBIC)")
for n in top_bin:
    plot_galaxy_decomposition(n, OUT_DIR / f"decomp_BINARY_{n}.png", "(top binary ΔBIC)")

print("Wrote decomposition PNGs for top 10 ternary and top 10 binary galaxies.")

In [ ]:
# --- Winner vs loser summary + KS + simple classifier ---

df["is_ternary_win"] = (df["delta_bic"] < WINNER_DBIC).astype(int)
feat_cols = [
    "Rmax",
    "Vmax",
    "n_points",
    "gas_frac_proxy",
    "disk_dom_proxy",
    "r_s3_over_Rmax",
    "sigma_s3_over_Rmax",
    "Vs3_over_Vmaxsq",
    "chi2_bin_per_n",
    "chi2_ter_per_n",
    "delta_chi2",
]

win = df[df["is_ternary_win"] == 1]
lose = df[df["is_ternary_win"] == 0]

ks_rows = []
for c in feat_cols:
    a, b2 = win[c].dropna(), lose[c].dropna()
    if len(a) > 2 and len(b2) > 2:
        stat, p = stats.ks_2samp(a, b2)
        ks_rows.append({"feature": c, "ks": stat, "p": p})
ks_df = pd.DataFrame(ks_rows).sort_values("p")
ks_df.to_csv(OUT_DIR / "winner_vs_loser_ks.csv", index=False)
print(ks_df.to_string(index=False))

try:
    from sklearn.tree import DecisionTreeClassifier

    X = df[feat_cols].fillna(df[feat_cols].median()).values
    y = df["is_ternary_win"].values
    clf = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
    clf.fit(X, y)
    imp = pd.DataFrame({"feature": feat_cols, "importance": clf.feature_importances_}).sort_values(
        "importance", ascending=False
    )
    imp.to_csv(OUT_DIR / "classifier_importances.csv", index=False)
    print(imp.to_string(index=False))
except ImportError:
    print("sklearn not installed — skip classifier")

## Stronger baselines (Einasto, MOND / RAR)

**Burkert** halo is included in `baseline_halos.csv`. **Einasto** and full **MOND/RAR** baselines usually need numerical enclosed-mass integration or a dynamics library (e.g. `galpy`, `AGAMA`) plus consistent M/L assumptions from SPARC. This notebook stays dependency-light; treat Einasto/MOND as **documented extensions** when you are ready to match published SPARC pipelines.

In [ ]:
# Quick summary for verdict narrative
# delta_bic = ternary_bic - binary_bic (negative => ternary wins overall)
# For disk vs random anchor: if disk coupling helps, random anchor ternary should
# usually be *less* favorable (higher delta_bic) than disk-anchored ternary.
m = adv_df.merge(df[["name", "winner", "delta_bic"]], on="name", how="left")
tw = m["winner"] == "TERNARY"
if tw.sum():
    sub = m.loc[tw]
    print("Among ternary winners (ΔBIC < -2):")
    print("  frac(random-anchor ternary LESS favorable than disk):", (sub["delta_bic_random_anchor"] > sub["delta_bic"]).mean())
    print("  frac(mirrored-anchor LESS favorable than disk):", (sub["delta_bic_mirrored_anchor"] > sub["delta_bic"]).mean())
    print("  frac(free-Gaussian ternary MORE favorable than disk-ternary):", (sub["delta_bic_free_gauss"] < sub["delta_bic"]).mean())
if len(base_df):
    mb = base_df.merge(df[["name", "winner"]], on="name")
    twb = mb["winner"] == "TERNARY"
    if twb.sum() and mb["bic_Burkert"].notna().any():
        print("  frac(Burkert BIC < disk-ternary BIC) among ternary winners:", (mb.loc[twb, "bic_Burkert"] < mb.loc[twb, "bic_NFW_ternary_disk"]).mean())

## Verdict rubric (A / B / C / D)

Use **evidence in the CSVs and figures above**, not vibes:

| Code | Meaning | Typical evidence |
|------|---------|------------------|
| **A** | S₃ structured & disk-coupled | Permutation p low; random/mirrored anchors **rarely** beat true coupling; free Gaussian **does not** uniformly destroy interpretation of disk anchor |
| **B** | S₃ mainly fixes NFW failure | Burkert (or other cored halo) **often** matches ternary χ² with **equal or fewer** parameters; NFW residuals systematic |
| **C** | S₃ generic overfitting | Free Gaussian **beats** disk-ternary often; random anchors competitive; classifier meaningless |
| **D** | Inconclusive | Conflicting signals; small winner *n*; controls unstable |

**Einasto / full MOND-RAR:** extension hooks — add numerical Einasto mass integration or McGaugh-style RAR if you need stronger baselines for publication.

---

### Your stated numbers (sanity check)

After running all cells, compare `df['winner'].value_counts()` to:

- 143 galaxies, ~21 ternary / ~102 binary / ~20 ties, median ΔBIC ≈ +2.5, mean ΔBIC ≈ −6.8.

If counts diverge, the fit implementation or data version differs from the archived Colab run — treat this notebook as the **reproducible reference** going forward.